# 02 — Frozen baseline smoke training
Run notebook 00 first and keep the same Colab GPU kernel. This notebook builds the standard 4,040-image training manifest, then runs the required 32-image/two-epoch smoke job for both frozen backbones. No test set is used.

In [ ]:
# Explicit Drive dataset locations — edit these four paths only if yours differ.
CAMO_TRAIN_IMAGES = '/content/drive/MyDrive/cod-ssl/data/CAMO/Train/Image'
CAMO_TRAIN_MASKS = '/content/drive/MyDrive/cod-ssl/data/CAMO/Train/GT'
COD10K_TRAIN_IMAGES = '/content/drive/MyDrive/cod-ssl/data/COD10K/Train/Image'
COD10K_TRAIN_MASKS = '/content/drive/MyDrive/cod-ssl/data/COD10K/Train/GT_Object'
PROJECT_DIR = '/content/cod-ssl'
RUNS_ROOT = '/content/drive/MyDrive/cod-ssl/runs'
SAMPLE_IMAGE = '/content/cod_ssl_sample_image.png'

In [ ]:
# Restore environment variables if this notebook was attached to a fresh kernel.
import os
os.environ.setdefault('DINOV3_REPO_DIR','/content/third_party/dinov3')
os.environ.setdefault('DINOV3_WEIGHTS','/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth')
os.environ.setdefault('VJEPA2_REPO_DIR','/content/third_party/vjepa2')
os.environ.setdefault('VJEPA21_WEIGHTS','/content/drive/MyDrive/cod-ssl/checkpoints/vjepa2_1_vitb_dist_vitG_384.pt')
from pathlib import Path
import torch
required=[PROJECT_DIR,CAMO_TRAIN_IMAGES,CAMO_TRAIN_MASKS,COD10K_TRAIN_IMAGES,COD10K_TRAIN_MASKS,os.environ['DINOV3_WEIGHTS'],os.environ['VJEPA21_WEIGHTS']]
missing=[path for path in required if not Path(path).exists()]
if missing: raise FileNotFoundError('Missing required paths:\n'+'\n'.join(missing))
if not torch.cuda.is_available(): raise RuntimeError('Connect to a GPU-backed Colab kernel.')
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
# Pull the latest project and reinstall it in the current kernel.
import subprocess, sys
subprocess.run(['git','-C',PROJECT_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{PROJECT_DIR}[dev,notebooks]'],check=True)

In [ ]:
# Create source manifests using explicit directories; no layout assumptions enter training code.
manifest_dir=Path(PROJECT_DIR)/'manifests'
camo_manifest=manifest_dir/'camo_train.csv'
cod10k_manifest=manifest_dir/'cod10k_train.csv'
for source,images,masks,output in [('camo',CAMO_TRAIN_IMAGES,CAMO_TRAIN_MASKS,camo_manifest),('cod10k',COD10K_TRAIN_IMAGES,COD10K_TRAIN_MASKS,cod10k_manifest)]:
    subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/prepare_manifests.py','--source',source,'--images',images,'--masks',masks,'--output',str(output),'--image-glob','*','--mask-suffix','.png'],cwd=PROJECT_DIR,check=True)
import pandas as pd
camo=pd.read_csv(camo_manifest); cod10k=pd.read_csv(cod10k_manifest)
if len(camo)!=1000: raise ValueError(f'CAMO train must contain 1000 images, found {len(camo)}')
if len(cod10k)!=3040: raise ValueError(f'COD10K train must contain 3040 images, found {len(cod10k)}')
train_all=pd.concat([camo,cod10k],ignore_index=True)
if train_all.image_path.duplicated().any(): raise ValueError('Duplicate training image paths found.')
train_manifest=manifest_dir/'train_all.csv'
train_all.to_csv(train_manifest,index=False)
print(train_all.groupby('source').size()); print('Total:',len(train_all))

In [ ]:
# Ensure the automatically downloaded smoke-test image exists for checkpoint reload verification.
from urllib.request import urlretrieve
if not Path(SAMPLE_IMAGE).is_file():
    urlretrieve('https://raw.githubusercontent.com/DengPingFan/SINet/master/Images/CamouflagedTask.png',SAMPLE_IMAGE)
print('Sample image:',SAMPLE_IMAGE)

In [ ]:
# Run frozen DINOv3 + common decoder on exactly 32 images for two epochs.
dino_before=set(Path(RUNS_ROOT).glob('*_dinov3_vitb16_seed42'))
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/train.py','--config',f'{PROJECT_DIR}/configs/frozen_dinov3_vitb16.yaml','--runs-root',RUNS_ROOT,'--limit-train','32','--epochs','2'],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'DINOv3 smoke training failed: {result.returncode}')
dino_new=set(Path(RUNS_ROOT).glob('*_dinov3_vitb16_seed42'))-dino_before
if len(dino_new)!=1: raise RuntimeError(f'Could not identify DINOv3 run: {dino_new}')
DINO_RUN=str(dino_new.pop()); print('DINO_RUN=',DINO_RUN)

In [ ]:
# Reload the DINOv3 checkpoint and verify finite 384×384 logits and the freeze invariant.
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/verify_checkpoint.py','--run',DINO_RUN,'--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError('DINOv3 checkpoint reload failed.')

In [ ]:
# Release process-local allocator caches before V-JEPA.
import gc
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Run frozen V-JEPA 2.1 + the identical common decoder and training settings.
vjepa_before=set(Path(RUNS_ROOT).glob('*_vjepa21_vitb16_seed42'))
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/train.py','--config',f'{PROJECT_DIR}/configs/frozen_vjepa21_vitb16.yaml','--runs-root',RUNS_ROOT,'--limit-train','32','--epochs','2'],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError(f'V-JEPA smoke training failed: {result.returncode}')
vjepa_new=set(Path(RUNS_ROOT).glob('*_vjepa21_vitb16_seed42'))-vjepa_before
if len(vjepa_new)!=1: raise RuntimeError(f'Could not identify V-JEPA run: {vjepa_new}')
VJEPA_RUN=str(vjepa_new.pop()); print('VJEPA_RUN=',VJEPA_RUN)

In [ ]:
# Reload the V-JEPA checkpoint and verify its output and freeze invariant.
result=subprocess.run([sys.executable,f'{PROJECT_DIR}/scripts/verify_checkpoint.py','--run',VJEPA_RUN,'--image',SAMPLE_IMAGE],cwd=PROJECT_DIR,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout)
if result.returncode: raise RuntimeError('V-JEPA checkpoint reload failed.')

In [ ]:
# Summarize smoke losses and required artifacts.
import numpy as np
for name,run in [('DINOv3',DINO_RUN),('V-JEPA 2.1',VJEPA_RUN)]:
    log=pd.read_csv(Path(run)/'training_log.csv')
    if not np.isfinite(log.loss).all(): raise RuntimeError(f'{name} produced non-finite loss')
    required=[Path(run)/'checkpoints/last.pt',Path(run)/'samples/training_sample.png',Path(run)/'config.yaml',Path(run)/'environment.txt',Path(run)/'upstream_versions.json']
    missing=[str(path) for path in required if not path.is_file()]
    if missing: raise FileNotFoundError(f'{name} missing artifacts: {missing}')
    print(f'\n{name}: {run}'); print(log[['epoch','loss','learning_rate','wall_time_seconds']].to_string(index=False))
    print('Prediction:',Path(run)/'samples/training_sample.png')

Milestone H passes when both jobs have finite losses, saved sample predictions, reloadable checkpoints, `[1,1,384,384]` finite logits, and zero trainable/gradient-bearing backbone parameters. Full experiments remain blocked until the smoke outputs are visually inspected.